# Estimating hallmark pathway expression from the ATAC component of dATAC and defining epigenetic classes

## Import R libraries

In [2]:
suppressPackageStartupMessages({
    library(ArchR)
    library(msigdbr)
    library(GSVA)
    library(readr)
    library(stringr)
    library(readxl)
    library(stats)
    library(fastcluster)
    library(mclust)
    library(Matrix)#
    library(pbapply)#
    library(ggplot2)#
    library(dplyr) #
    library(ComplexHeatmap)#
    library(reshape2)#
    library(reticulate)#
    library(data.table)#
})


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

## Subset ArchR project with all cells for those with high quality CNA and ATAC profiles

In [3]:
# load the project with all cells
proj <- loadArchRProject(path = sprintf("../Data/ArchR_Projects/Save_SW620_ATAC_DATAC_Results"))

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [5]:
# ids of all cells
ids <- proj$cellNames

# for HQ datac cells only - filter for these cells
hq_d_cells <- data.frame(read_tsv("../Data/CNA_Data/Processed/CNA_Profiles_Segmented.tsv"))
hq_d_cells <- unique(hq_d_cells$cell_id)

# Cell ids need to be modified to match those in the ArchR project
# First, replace dsiv cell ids with the corresponding barcode
dsiv_barcodes <- read_csv("../Data/Other/SW620bc_DSIV_barcode_matching.csv",
                          col_types = "cc") 
cell_ids_mod <- hq_d_cells
for (id in hq_d_cells){
  if (id %in% dsiv_barcodes$sample_name){
    cell_ids_mod[cell_ids_mod == id] = dsiv_barcodes[dsiv_barcodes$sample_name == id,]$cell_barcode
  }
}

# Second, for each cell, find its corresponding index in the cell ids vector "ids" (i.e. column names of the gene score matrix)
id_inds <- c()
for (id in cell_ids_mod){
  if (length(grep("DSIVa",id))>0 || length(grep("DSIVb",id))>0){
    ind <- grep(id,ids)
    id_inds <- rbind(id_inds,ind)
  } else {
    ind <- grep(sprintf("%s_",id),ids)
    id_inds <- rbind(id_inds,ind)
  }
}

# Subset the column names of the gene score matrix to get the ids of cells that have passed ATAC and CNA QCs

ids_incl <- ids[id_inds]

# Subset the ArchR project for cells that pass the CNA and ATAC QC
proj_fil <- subsetArchRProject(
  ArchRProj = proj,
  cells = ids_incl,
  outputDirectory = "../Data/ArchR_Projects/Save_SW620_DATAC_Results",
  dropCells = TRUE,
  logFile = NULL,
  threads = getArchRThreads(),
  force = TRUE)


proj_fil <- loadArchRProject("../Data/ArchR_Projects/Save_SW620_DATAC_Results")


Rows: 34068 Columns: 5
── Column specification ─────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): cell_id, chrom
dbl (3): start, end, state

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Copying ArchRProject to new outputDirectory : /Users/ccolson/Documents/LongTermResistance_CellLineWork/dATACManuscript_AssociatedCode_Revision/Data/ArchR_Projects/Save_SW620_DATAC_Results

Copying Arrow Files...

.copyArrow : Initializing Out ArrowFile

.copyArrow : Adding Metadata to Out ArrowFile

.copyArrow : Adding SubMatrices to Out ArrowFile

.copyArrow : Initializing Out ArrowFile

.copyArrow : Adding Metadata to Out ArrowFile

.copyArrow : Adding SubMatrices to Out ArrowFile

.copyArrow : Initializing Out ArrowFile

.copyArrow : Adding Metadata to Out ArrowFile

.copyArrow : Adding SubMatrices to Out ArrowFile

.copyArrow : Initializing Out ArrowFile

.copy

## Define pathways of interest

In [7]:
# Retrieve hallmark pathways from msigdb
h_gene_sets <-  msigdbr(species = "human", category = "H")

# Remove unrelated pathways
h_gene_sets = h_gene_sets[!h_gene_sets$gs_name %in% c("HALLMARK_SPERMATOGENESIS", "HALLMARK_MYOGENESIS","HALLMARK_PANCREAS_BETA_CELLS"),]

# Retrieve the WNT-signalling pathways and add it to the dataframe of gene sets
WNT <- msigdbr(species = "human",
               category = "C2",
               subcategory = "CP")[msigdbr(species = "human",
                                           category = "C2",
                                           subcategory = "CP")$gs_name == "WNT_SIGNALING",]
h_gene_sets = rbind(h_gene_sets,WNT)

# Retrieve the intestinal stem cell pathway and add it to the dataframe of gene sets
ISC <- readxl::read_excel("../Data/Other/Intestinal_Stem_Cell.xls", sheet = "ISC genes", range = "B5:B76")
ISC <- ISC %>% dplyr::mutate_at("Gene", str_replace,"\\*", "") %>%
               dplyr::rename("gene_symbol" = "Gene") %>%
               dplyr::mutate(gs_name = "INTESTINAL_STEM_CELL") %>%
               dplyr::relocate(gs_name) 

h_gene_sets <- h_gene_sets %>% dplyr::select(gs_name, gene_symbol)
h_gene_sets <- rbind(h_gene_sets,ISC)

# Remove duplicated gene symbols
h_gene_sets <- h_gene_sets %>% distinct()

# Retain only genes in the gene sets that have ArchR gene scores
h_gene_sets = h_gene_sets[h_gene_sets$gene_symbol %in% getFeatures(proj_fil),]

# Create the pathway list
features <- list()
pathways <- unique(h_gene_sets$gs_name)
for (i in (pathways)){
  features <- append(features, list(c(h_gene_sets[h_gene_sets$gs_name == i,]$gene_symbol)))
}
names(features) <- pathways

# comment out the below command if this rds has already been created
saveRDS(features, "../Data/ATAC_Data/GeneExpressionScores/HallmarkPathways.rds")

## Obtain the gene score matrix from ArchR for the cells that pass ATAC and CNA QCs and the genes in relevant pathways

In [8]:
# Extract the gene score matrix from the ArchR project
genes <- getMatrixFromProject(proj_fil, useMatrix = "GeneScoreMatrix")
gene_mat <- genes@assays@data@listData$GeneScoreMatrix

# Filter the gene score matrix for genes involved in the hallmark pathways

# Genes in unfiltered gene score matrix
gene_names <- genes@elementMetadata$name

# For every gene included in the pathways, define its row index in the gene score matrix and its name
gene_inds <- which(gene_names %in% h_gene_sets$gene_symbol)
genes_names_fil <- gene_names[gene_inds]

# Filter the gene score matrix
gene_mat_fil <- gene_mat[gene_inds,]

# as data frame of the transpose
  gene_df <- as.data.frame(t(as.matrix(gene_mat_fil)))
  colnames(gene_df) <- genes_names_fil
  
  gene_df <- gene_df[,-which(colSums(gene_df) == 0)] 
  gene_df <- cbind("Cell_id" = rownames(gene_df),gene_df)
  
  write_csv(gene_df,"../Data/ATAC_Data/DATAC/GeneExpressionScores/DATAC_Gene_Zscores.csv")  

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-161b132058e1-Date-2026-02-23_Time-11-11-26.675855.log
If there is an issue, please report to github with logFile!

2026-02-23 11:11:34.341084 : Organizing colData, 0.128 mins elapsed.

2026-02-23 11:11:34.420356 : Organizing rowData, 0.129 mins elapsed.

2026-02-23 11:11:34.433362 : Organizing rowRanges, 0.129 mins elapsed.

2026-02-23 11:11:34.493493 : Organizing Assays (1 of 1), 0.13 mins elapsed.

2026-02-23 11:11:35.244475 : Constructing SummarizedExperiment, 0.143 mins elapsed.

2026-02-23 11:11:35.617726 : Finished Matrix Creation, 0.149 mins elapsed.



## Bootstrapping procedure to impute gene scores using MAGIC, compute pathway scores and cluster cells 

In [51]:
# Make sure MAGIC is loaded
# Import magic in python
import("magic")

# Import magic R library
library(Rmagic)

# ---------------- Define user parameters ----------------
input_csv       <- "../Data/ATAC_Data/GeneExpressionScores/DATAC/DATAC_Gene_Zscores.csv"    # rows=cells, cols=genes
pathways_rds    <- "HallmarkPathways.rds"      # optional RDS containing named list of gene sets

n_boot          <- 1000    # number of bootstraps
n_clusters      <- 4        # number of clusters for cutree
avg_duplicates  <- TRUE      # average duplicate cells in bootstraps
seed_base       <- 7
save_prefix     <- "../Data/ATAC_Data/GeneExpressionScores/DATAC/DATAC_ImputedPathwayScore_Bootstrapping/magic_gsva_bootstrap_zscoreParam"

set.seed(seed_base)

# ---------------- Load data ----------------
cat("Loading expression matrix...\n")
mat <- as.matrix(read.csv(input_csv, row.names = 1, check.names = FALSE))
rownames(mat) <- make.unique(rownames(mat))
cell_ids <- rownames(mat)
cat("Matrix dimensions:", nrow(mat), "cells ×", ncol(mat), "genes\n")

# ---------------- Load pathways ----------------
if (exists("pathways_list")) {
  pathways_list <- get("pathways_list")
  cat("Using 'pathways_list' from environment; num pathways:", length(pathways_list), "\n")
} else if (!is.null(pathways_rds) && file.exists(pathways_rds)) {
  pathways_list <- readRDS(pathways_rds)
  cat("Loaded pathways_list from", pathways_rds, " (# pathways:", length(pathways_list), ")\n")
} else {
  stop("No 'pathways_list' found.")
}

# ---------------- Define helper functions ----------------
run_magic_impute <- function(mat_sub, seed = NULL) {
  mg <- magic(mat_sub, t = 2, seed = seed)
  as.matrix(mg$result)
}

compute_gsva_scores_zscoreParam <- function(expr_mat, pathways_list) {
  zPar <- zscoreParam(t(as.matrix(expr_mat)), geneSets = pathways_list)
  gsva.es <- gsva(zPar, verbose = FALSE)
  t(gsva.es)
}

zscore_matrix <- function(mat) {
  z <- scale(mat, center = TRUE, scale = TRUE)
  z[is.na(z)] <- 0
  z
}

cluster_hier_pathway_raw <- function(pathway_z, K = n_clusters, scale = FALSE) {
  mat <- if (scale) zscore_matrix(pathway_z) else pathway_z # decide whether or not to rescale what comes out of GSVA
  d <- dist(mat, method = "euclidean")
  hc <- fastcluster::hclust(d, method = "ward.D2")
  cl <- cutree(hc, k = K)
  #names(cl) <- rownames(mat)
}

# ---------------- Full-data reference ----------------
cat("Running MAGIC on full data (reference)...\n")
mg_full <- run_magic_impute(mat, seed = seed_base)
mg_full_z <- zscore_matrix(mg_full)
gsva_full <- compute_gsva_scores_zscoreParam(mg_full_z, pathways_list)
ref_clusters <- cluster_hier_pathway_raw(gsva_full, K = n_clusters, scale = FALSE)

# ---------------- Run bootstrap ----------------
n <- nrow(mat)
O <- matrix(0L, n, n, dimnames = list(cell_ids, cell_ids))
C <- matrix(0L, n, n, dimnames = list(cell_ids, cell_ids))

ARI_vec <- numeric(n_boot)

# For consensus pathway scores
consensus_pathway_sum <- matrix(0, nrow = n, ncol = length(pathways_list),
                                dimnames = list(cell_ids, names(pathways_list)))
counts <- matrix(0, nrow = n, ncol = length(pathways_list),
                 dimnames = list(cell_ids, names(pathways_list)))

pb <- txtProgressBar(min = 0, max = n_boot, style = 3)
for (b in seq_len(n_boot)) {
  setTxtProgressBar(pb, b)
  seed_b <- seed_base + b
  set.seed(seed_b)
  
  # Sample cells with replacement
  sample_idx <- sample(seq_len(n), size = n, replace = TRUE)
  sampled_ids <- cell_ids[sample_idx]
  mat_sub <- mat[sampled_ids, , drop = FALSE]
  rownames(mat_sub) <- sampled_ids
  
  
  # MAGIC
  imputed_sub <- run_magic_impute(mat_sub, seed = seed_b)
  rownames(imputed_sub) <- sub("\\.", "#", rownames(imputed_sub))
  
  # Average duplicates if requested
  if (avg_duplicates) {
    uniq_ids <- unique(sampled_ids)
    if (length(uniq_ids) < length(sampled_ids)) {
      agg_mat <- sapply(uniq_ids, function(id)
        colMeans(imputed_sub[grep(sprintf("%s(\\.\\d+)?$",id),rownames(imputed_sub)), , drop = FALSE]))
      imputed_for_scores <- t(agg_mat)
    } else {
      imputed_for_scores <- imputed_sub
    }
  } else {
    imputed_for_scores <- imputed_sub
  }
  
  # GSVA on imputed gene scores
  imputed_for_scores_z <- zscore_matrix(imputed_for_scores)
  gsva_res <- compute_gsva_scores_zscoreParam(imputed_for_scores_z, pathways_list)
  pathway_z <- gsva_res 
  
  # Update consensus pathway sum
  for (i in seq_along(rownames(pathway_z))) {
    consensus_pathway_sum[rownames(pathway_z)[i], ] <- consensus_pathway_sum[rownames(pathway_z)[i], ] +
      pathway_z[i, ]
    counts[rownames(pathway_z)[i], ] <- counts[rownames(pathway_z)[i], ] + 1
  }
  
  # Cluster
  clusters <- cluster_hier_pathway_raw(pathway_z, K = n_clusters, scale = FALSE)
  
  uniq_ids <- names(clusters)
  idx <- match(uniq_ids, cell_ids)
  for (i in seq_along(idx)) {
    for (j in seq_along(idx)) {
      O[idx[i], idx[j]] <- O[idx[i], idx[j]] + 1L
      if (clusters[i] == clusters[j]) C[idx[i], idx[j]] <- C[idx[i], idx[j]] + 1L
    }
  }

  # Compute the adjusted rand index to quantify clustering similarity between each boostrap cluster and the reference cluster                      
  common <- intersect(uniq_ids, names(ref_clusters))
  if (length(common) > 1) {
    ARI_vec[b] <- adjustedRandIndex(clusters[common], ref_clusters[common])
  } else {
    ARI_vec[b] <- NA
  }
}
close(pb)
cat("\nBootstrapping complete.\n")

# ---------------- Consensus pathway scores ----------------
consensus_pathway_scores <- consensus_pathway_sum / counts

# ---------------- Compute co-assignment probabilities ----------------
coassign_prob <- C / pmax(O, 1)


# ---------------- Consensus clustering ----------------
dist_cons <- as.dist(1 - coassign_prob)
hc_cons <- fastcluster::hclust(dist_cons, method = "ward.D2")
consensus_clusters <- cutree(hc_cons, k = n_clusters)
names(consensus_clusters) <- cell_ids

# ---------------- Compute per-cell stability scores ----------------

min_cooccur <- 3  # set to 0 to disable thresholding

cell_stability <- sapply(cell_ids, function(cid) {
  cl <- consensus_clusters[cid]
  members <- names(consensus_clusters)[consensus_clusters == cl]
  # exclude self
  members_no_self <- setdiff(members, cid)
  if (length(members_no_self) == 0) return(NA_real_)       # singleton cluster
  
  # optionally require minimum co-occurrence count
  if (min_cooccur > 0) {
    valid_members <- members_no_self[vapply(members_no_self, function(m) {
      O[cid, m] >= min_cooccur
    }, logical(1))]
  } else {
    valid_members <- members_no_self
  }
  if (length(valid_members) == 0) return(NA_real_)         # not enough info
  
  mean(coassign_prob[cid, valid_members], na.rm = TRUE)
})
names(cell_stability) <- cell_ids

# Cluster-level summary: mean stability per consensus cluster (ignore NA cells)
cluster_stability <- tapply(cell_stability, consensus_clusters, function(x) mean(x, na.rm = TRUE))

# ---------------- Summaries ----------------
cat("Mean cell stability:", round(mean(cell_stability, na.rm = TRUE), 3), "\n")
cat("Mean ARI vs full clustering:", round(mean(ARI_vec, na.rm = TRUE), 3), "\n")

# ---------------- Save outputs ----------------
out <- list(
  params = list(n_boot = n_boot, n_clusters = n_clusters, seed = seed_base),
  pathways_list = pathways_list,
  ARI_vec = ARI_vec,
  coassign_prob = coassign_prob,
  cell_stability = cell_stability,
  cluster_stability = cluster_stability,
  consensus_clusters = consensus_clusters,
  ref_clusters = ref_clusters,
  pathway_z_full = gsva_full,
  consensus_pathway_scores = consensus_pathway_scores
)

saveRDS(out, file = paste0(save_prefix, ".rds"))
write.csv(data.frame(cell_id = names(consensus_clusters),
                     consensus_cluster = consensus_clusters,
                     stability = cell_stability),
          file = paste0(save_prefix, "_consensus_clusters.csv"), row.names = FALSE)
write.csv(data.frame(ARI = ARI_vec),
          file = paste0(save_prefix, "_ARI_per_bootstrap.csv"), row.names = FALSE)
write.csv(consensus_pathway_scores,
          file = paste0(save_prefix, "_consensus_pathway_scores.csv"), row.names = TRUE)
